# 02 — Feature Engineering
Builds aggregated, lag, first, 3M, 6M, and diff features from preprocessed data.

In [1]:
# ── Imports & Load ───────────────────────────────────────
import pandas as pd
import numpy as np
import gc
import warnings
warnings.filterwarnings("ignore")
df = pd.read_parquet('../data/processed/train_preprocessed.parquet')
print(f'Shape: {df.shape}')
print(f'Memory: {df.memory_usage(deep=True).sum() / 1e9:.2f} GB')

Shape: (2766578, 167)
Memory: 2.13 GB


In [2]:
# ── Define Feature Groups ────────────────────────────────
continuous_features = df.select_dtypes(include='float32').columns.tolist()
cat_cols = [
    'D_63', 'D_64', 'B_30', 'B_38',
    'D_114', 'D_116', 'D_117',
    'D_120', 'D_126', 'D_68'
]

print(f'Continuous features: {len(continuous_features)}')
print(f'Categorical features: {len(cat_cols)}')

Continuous features: 154
Categorical features: 10


### Customer-Level Aggregation (Continuous Features)

Each customer has multiple monthly statements. To create a single training record per customer, continuous features were aggregated across the customer's transaction history using the following summary statistics:

- **Mean** – Average historical value.
- **Standard Deviation** – Measures variability over time.
- **Minimum** – Lowest observed value.
- **Maximum** – Highest observed value.
- **Last** – Most recent observed value, capturing the customer's latest financial behaviour.
- **Median** – Robust measure of central tendency that is less sensitive to outliers.

These aggregated statistics summarize each customer's historical financial behaviour while preserving information about central tendency, variability, extreme values, and recent trends.

In [3]:
# ── Aggregated Features (Continuous) ────────────────────
# mean, std, min, max, last, median per customer across all statements
agg_dict = {col: ['mean', 'std', 'min', 'max', 'last', 'median']
            for col in continuous_features}
df_agg = df.groupby('customer_ID').agg(agg_dict)
df_agg.columns = ['_'.join(col) for col in df_agg.columns]
df_agg = df_agg.reset_index()
print(f'Aggregated (continuous): {df_agg.shape}')

Aggregated (continuous): (229456, 925)


### Customer-Level Aggregation (Categorical Features)

Categorical features were aggregated at the customer level to summarize historical categorical behaviour across multiple monthly statements. Three aggregation strategies were applied:

- **Count** – Number of non-missing observations available for each customer.
- **Last** – Most recent observed category, capturing the latest customer status.
- **Number of Unique Categories (`nunique`)** – Measures how frequently a customer's categorical value changed over time.

These aggregated features transform the sequential categorical data into a single customer-level representation while preserving information about recency, data availability, and category transitions.

In [4]:
# ── Aggregated Features (Categorical) ───────────────────
# count, last, nunique per customer
cat_agg = df.groupby('customer_ID')[cat_cols].agg(['count', 'last', 'nunique'])
cat_agg.columns = ['_'.join(col) for col in cat_agg.columns]
cat_agg = cat_agg.reset_index()
print(f'Aggregated (categorical): {cat_agg.shape}')

Aggregated (categorical): (229456, 31)


### Lag Features (Last − Mean)

To capture recent behavioral changes, lag features were created by subtracting the historical mean from the most recent value for each continuous feature.

**Lag = Last Value − Historical Mean**

A positive lag indicates the customer's latest observation is above their historical average, while a negative lag indicates it is below their typical behaviour. These features help capture recent changes that may signal increasing or decreasing credit risk.

In [5]:
# ── Lag Features (last - mean) ───────────────────────────
# Captures recent behavioral changes per feature
lag_df = pd.DataFrame()
lag_df['customer_ID'] = df_agg['customer_ID']
for col in continuous_features:
    lag_df[col + '_lag'] = df_agg[col + '_last'] - df_agg[col + '_mean']
print(f'Lag features: {lag_df.shape}')

Lag features: (229456, 155)


### First Observed Values

The first observed values capture each customer's baseline financial profile and complement the most recent observations used elsewhere in the feature engineering pipeline.

In [6]:
# ── First Values ─────────────────────────────────────────
# First statement value per feature per customer
first_df = df.groupby('customer_ID')[continuous_features].first()
first_df.columns = [col + '_first' for col in first_df.columns]
first_df = first_df.reset_index()
print(f'First features: {first_df.shape}')

First features: (229456, 155)


### Recent 3-Month Features

To capture recent customer behaviour, continuous features were aggregated using the **last three monthly statements**. Summary statistics (mean, standard deviation, minimum, maximum, and last value) were computed to represent short-term financial trends that may be predictive of future default.

In [7]:
# ── 3M Features (Last 3 Statements) ─────────────────────
# Captures most recent short-term behavior
df_3m = df.groupby('customer_ID').tail(3)
agg_3m = df_3m.groupby('customer_ID')[continuous_features].agg(['mean', 'std', 'min', 'max', 'last'])
agg_3m.columns = ['_'.join(col) + '_3m' for col in agg_3m.columns]
agg_3m = agg_3m.reset_index()
print(f'3M features: {agg_3m.shape}')

3M features: (229456, 771)


### Recent 6-Month Features
To capture medium-term customer behaviour, continuous features were aggregated using the **last six monthly statements**. Summary statistics (mean, standard deviation, minimum, maximum, and last value) were computed to better capture recent financial trends that may be predictive of future default.

In [8]:
# ── 6M Features (Last 6 Statements) ─────────────────────
# Captures medium-term behavioral trends
df_6m = df.groupby('customer_ID').tail(6)
agg_6m = df_6m.groupby('customer_ID')[continuous_features].agg(['mean', 'std', 'min', 'max', 'last'])
agg_6m.columns = ['_'.join(col) + '_6m' for col in agg_6m.columns]
agg_6m = agg_6m.reset_index()
print(f'6M features: {agg_6m.shape}')

6M features: (229456, 771)


### Month-over-Month Difference Features

To capture changes in customer behaviour over time, month-over-month differences were computed for each continuous feature. The resulting differences were then aggregated using the **mean**, **standard deviation**, **minimum**, and **maximum** to summarize the magnitude and variability of changes throughout the customer's history.

In [9]:
# ── Diff Features ────────────────────────────────────────
# Month-over-month changes — captures rate of deterioration
df_sorted = df.sort_values(['customer_ID', 'S_2'])
diff_df = df_sorted.groupby('customer_ID')[continuous_features].diff()
diff_df['customer_ID'] = df_sorted['customer_ID'].values
diff_agg = diff_df.groupby('customer_ID').agg(['mean', 'std', 'min', 'max'])
diff_agg.columns = ['_'.join(col) + '_diff' for col in diff_agg.columns]
diff_agg = diff_agg.reset_index()
print(f'Diff features: {diff_agg.shape}')
gc.collect()

Diff features: (229456, 617)


0

In [10]:
# ── Merge All Features ───────────────────────────────────
df_features = df_agg.merge(cat_agg, on='customer_ID')
df_features = df_features.merge(lag_df, on='customer_ID')
df_features = df_features.merge(first_df, on='customer_ID')
df_features = df_features.merge(agg_3m, on='customer_ID')
df_features = df_features.merge(agg_6m, on='customer_ID')
df_features = df_features.merge(diff_agg, on='customer_ID')
print(f'Final shape: {df_features.shape}')
print(f'Memory: {df_features.memory_usage(deep=True).sum() / 1e9:.2f} GB')

Final shape: (229456, 3419)
Memory: 3.18 GB


In [11]:
# ── Merge Target & Save ──────────────────────────────────
labels = pd.read_csv('../data/raw/train_labels.csv')
df_features = df_features.merge(labels, on='customer_ID')
print(f'Final shape with target: {df_features.shape}')

df_features.to_parquet('../data/processed/train_features.parquet', index=False)
print('Saved!')

Final shape with target: (229456, 3420)
Saved!


Customer-level feature engineering transformed the original monthly transaction records into a single record per customer. The final dataset contains 229,456 customers and 3,418 engineered features, combining historical aggregations, recent behaviour, temporal change features, and categorical summaries. This feature matrix was used for training the machine learning models.